# 01 — Dataset definitivo para pronóstico 2026

Construye EVA municipal de papa, descarga NASA POWER por celda, genera indicadores climáticos y materializa el dataset sin producción ni área cosechada como predictores.

In [ ]:
from pathlib import Path
import subprocess
import sys

try:
    from google.colab import drive
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    drive.mount('/content/drive')
    REPO_ROOT = Path('/content/suelosabio')
    if not (REPO_ROOT / '.git').exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'feature/SCRUM-17',
            'https://github.com/cybercolombia/suelosabio.git', str(REPO_ROOT)
        ], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '-r',
        str(REPO_ROOT / 'notebooks/CropForecasting/requirements.txt')
    ], check=True)
else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / '.git').exists():
        REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))
REPO_ROOT

In [ ]:
from notebooks.CropForecasting.config import load_forecasting_config
from notebooks.CropForecasting.run_pipeline import build_all

CONFIG = load_forecasting_config(
    in_colab=EN_COLAB,
    mount_drive=False,
    create_outputs=True,
)
CONFIG

## Construcción reproducible

La bandera evita descargas accidentales al abrir el notebook. Al activarla se ejecuta el pipeline completo y reanudable. Las 17 celdas NASA ya descargadas se reutilizan.

In [ ]:
EJECUTAR_PIPELINE = False

if EJECUTAR_PIPELINE:
    RESULTADO = build_all(in_colab=EN_COLAB)
    print('Dataset y evaluación completados.')
else:
    print('Ejecución desactivada; se leerán los artefactos existentes.')

In [ ]:
import json
import pandas as pd

DATASET_ROOT = CONFIG.dataset_root
dataset = pd.read_parquet(DATASET_ROOT / 'dataset_definitivo.parquet')
municipios = pd.read_parquet(DATASET_ROOT / 'municipios_objetivo.parquet')
manifest = json.loads((DATASET_ROOT / 'manifest.json').read_text(encoding='utf-8'))

display(pd.DataFrame([{
    'filas': len(dataset),
    'columnas': len(dataset.columns),
    'municipios': dataset.codigo_municipio.nunique(),
    'municipios_objetivo': len(municipios),
    'features_climaticas': len(manifest['climate_features']),
    'version': manifest['version'],
}]))
display(dataset.head())

In [ ]:
display(
    municipios[[
        'departamento', 'ranking_departamento', 'codigo_municipio',
        'municipio', 'area_sembrada_seleccion_ha', 'anios_seleccion'
    ]].sort_values(['departamento', 'ranking_departamento'])
)

display(dataset.groupby('anio').size().rename('filas').to_frame())
assert not dataset.duplicated(['codigo_municipio', 'anio', 'tipo_periodo', 'cultivo']).any()
assert dataset.loc[dataset.anio.eq(2026), 'rendimiento_t_ha'].isna().all()